# Vidéo vers 3DGS sur Colab T4
Exécuter les cellules dans l'ordre. **Tout le calcul reste dans `/content`** : vidéo copiée une fois, frames, masques, SfM, entraînement et logs. Drive contient les entrées et les sauvegardes uniquement.

**CUDA ne suffit pas : Spirula exige Vulkan NVIDIA.** Le contrôle bloque avant l'extraction si le pilote n'est pas utilisable. Installer `libvulkan1` ne fournit pas le pilote NVIDIA.

Les logs sont affichés en direct et conservés localement. Micromamba est lancé sans capture de sortie. `/content` disparaît à la fin du runtime : sauvegarder après une phase ou après un échec. Aucune restauration automatique de workspace Drive, aucun effacement d'installation existante.


In [ ]:
from pathlib import Path
import os, subprocess, sys

VIDEO_DRIVE = Path('/content/drive/MyDrive/gsplat/input/IMG_5017.MOV')
SPIRULA_ARCHIVE = Path('/content/drive/MyDrive/spirula-builds/spirula-studio-vulkan-20261114_203210.zip')
SPIRULA_BINARY_RELATIVE = 'build_vulkan/spirula'
SAM_MODEL_RELATIVE = 'models/sam3-q4_0.ggml'
NUM_FRAMES = 300
RUN_NAME = 'video_run'  # Nouveau nom si vidéo/images/réglages changent.
GIT_BRANCH = 'review_copilot'
ACCEPT_SAM_LICENSE = False  # Mettre True seulement après lecture et acceptation personnelle.
PHASE = 'all'  # 'prepare' : masques/SfM/découpage ; 'all' : jusqu'au splat.
SAVE_ON_SUCCESS = True

PROJECT = Path('/content/colmap_to_gs')
WORK = Path('/content/colmap_to_gs_work')
DRIVE_BACKUPS = Path('/content/drive/MyDrive/colmap_to_gs_runs')
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in ('.', '..'):
    raise ValueError('RUN_NAME doit être un simple nom de dossier.')
if not isinstance(NUM_FRAMES, int) or NUM_FRAMES <= 0 or PHASE not in ('all', 'prepare'):
    raise ValueError('NUM_FRAMES positif et PHASE all/prepare requis.')
RUN = WORK / RUN_NAME
OUTPUT = RUN / 'output'
FRAMES = RUN / 'frames'
LOGS = RUN / 'logs'
LOGS.mkdir(parents=True, exist_ok=True)
print('Calcul local :', RUN, '\nSauvegardes :', DRIVE_BACKUPS)


## 1. Drive et code
Le dépôt est mis à jour sans écraser de modifications locales. La branche choisie contient le notebook, son helper et l'environnement Conda.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
for path in (VIDEO_DRIVE, SPIRULA_ARCHIVE):
    if not path.is_file():
        raise FileNotFoundError(path)
if PROJECT.exists():
    if not (PROJECT / '.git').is_dir():
        raise RuntimeError(f'{PROJECT} existe sans dépôt Git ; choisir un autre PROJECT.')
    dirty = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True)
    if dirty.strip():
        raise RuntimeError('Dépôt modifié : sauvegarder les changements avant mise à jour.\n'+dirty)
    for args in (['fetch', 'origin', GIT_BRANCH+':refs/remotes/origin/'+GIT_BRANCH], ['checkout', GIT_BRANCH], ['pull', '--ff-only', 'origin', GIT_BRANCH]):
        subprocess.run(['git', '-C', str(PROJECT), *args], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', GIT_BRANCH, 'https://github.com/NicoIGN/colmap_to_gs.git', str(PROJECT)], check=True)
sys.path.insert(0, str(PROJECT / 'environment/google.colab'))
from colab_runtime import run_live, restore_spirula, save_snapshot, sha256
def run(command, name, **kwargs):
    return run_live(command, LOGS, name, **kwargs)
run(['git', '-C', str(PROJECT), 'submodule', 'update', '--init', '--recursive'], 'submodules')
run(['git', '-C', str(PROJECT), 'log', '-1', '--oneline'], 'version')


## 2. Dépendances et Vulkan : arrêter ici si NVIDIA n'est pas visible
Pas d'installation automatique de pilote NVIDIA : le pilote doit correspondre à celui du runtime Colab. Les erreurs `DISPLAY`/`XDG_RUNTIME_DIR` peuvent être liées au mode sans écran ; une erreur de création d'instance ou l'absence de GPU NVIDIA est bloquante.


In [ ]:
run(['nvidia-smi'], 'nvidia')
run(['apt-get', 'update', '-qq'], 'apt-update')
run(['apt-get', 'install', '-y', '--no-install-recommends', 'git-lfs', 'curl', 'bzip2', 'libvulkan1', 'vulkan-tools', 'libgl1'], 'apt-install', env={'DEBIAN_FRONTEND': 'noninteractive'})


### Diagnostic NVIDIA / Vulkan (avant tout calcul)
Cette cellule ne modifie ni les pilotes ni les variables d'environnement. Les commandes en échec sont signalées sans interrompre les autres diagnostics ; **cela ne valide pas Vulkan**. Le contrôle bloquant reste dans la cellule suivante.
Le rapport complet reste dans les logs locaux. Mettre `DIAGNOSTIC_LD_DEBUG=True` pour tracer aussi les bibliothèques réellement chargées : le rapport peut être volumineux, mais l'affichage est limité. Après un essai de runtime isolé, relancer cette cellule avec les mêmes variables que le calcul. Un ICD Vulkan SC n'est pas un ICD Vulkan standard.


In [ ]:
import json, shutil, tempfile
DIAGNOSTIC_LD_DEBUG = False

with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', prefix='vulkan-diagnostic-', suffix='.log', dir=LOGS, delete=False) as diagnostic_report:
    def diagnostic_print(text):
        print(text, flush=True)
        diagnostic_report.write(str(text)+'\n')
        diagnostic_report.flush()

    def diagnostic_command(command, *, child_env=None, loader_trace=False):
        diagnostic_print('\n=== '+ ' '.join(command)+' ===')
        if shutil.which(command[0]) is None:
            diagnostic_print('Commande indisponible : '+command[0])
            return ''
        try:
            result = subprocess.run(command, env=child_env, text=True, errors='replace', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=60)
        except subprocess.TimeoutExpired as error:
            partial = error.stdout or b''
            diagnostic_print(partial.decode(errors='replace') if isinstance(partial, bytes) else partial)
            diagnostic_print('Délai de 60 s dépassé ; diagnostic incomplet.')
            return ''
        diagnostic_report.write(result.stdout)
        diagnostic_report.flush()
        lines = result.stdout.splitlines()
        if loader_trace:
            lines = [line for line in lines if any(word in line.lower() for word in ('error', 'warning', 'undefined symbol', 'not found', 'devicename', 'devicetype'))]
        print('\n'.join(lines[:120]), flush=True)
        if len(lines) > 120:
            print('Affichage limité à 120 lignes ; sortie complète dans le rapport.', flush=True)
        diagnostic_print('Code de sortie : '+str(result.returncode))
        return result.stdout

    diagnostic_command(['nvidia-smi', '--query-gpu=name,driver_version', '--format=csv,noheader'])
    diagnostic_command(['dpkg-query', '-W', '-f=${Package} ${Version}\n', 'libnvidia-*'])
    diagnostic_print('\n=== Variables de sélection et de chargement ===')
    for key in sorted(os.environ):
        if key.startswith('VK_') or key in ('LD_LIBRARY_PATH', 'LD_PRELOAD', 'LIBGL_ALWAYS_SOFTWARE', 'NVIDIA_DRIVER_CAPABILITIES', 'NVIDIA_VISIBLE_DEVICES'):
            diagnostic_print(key+'='+os.environ[key])
    cache = diagnostic_command(['ldconfig', '-p'])
    diagnostic_print('\n=== Bibliothèques NVIDIA et dépendances GLX/EGL ===')
    libraries = {Path(line.split(' => ', 1)[1]) for line in cache.splitlines() if ' => ' in line and 'nvidia' in line.lower()}
    for directory in os.environ.get('LD_LIBRARY_PATH', '').split(':')+['/usr/lib64-nvidia', '/usr/local/nvidia/lib64', '/usr/lib/x86_64-linux-gnu']:
        if directory:
            libraries.update(Path(directory).glob('lib*nvidia*.so*'))
    if not libraries:
        diagnostic_print('Aucune bibliothèque NVIDIA trouvée dans les chemins inspectés.')
    for library in sorted(libraries):
        diagnostic_print(str(library)+' -> '+str(library.resolve()))
        if library.name.startswith(('libGLX_nvidia.so', 'libEGL_nvidia.so')):
            diagnostic_command(['ldd', str(library)])
    diagnostic_print('\n=== Descripteurs Vulkan ===')
    icds = set()
    for directory in ('/usr/share/vulkan/icd.d', '/etc/vulkan/icd.d', '/usr/local/share/vulkan/icd.d'):
        icds.update(Path(directory).glob('*.json'))
    for key in ('VK_DRIVER_FILES', 'VK_ICD_FILENAMES', 'VK_ADD_DRIVER_FILES'):
        icds.update(Path(path) for path in os.environ.get(key, '').split(':') if path)
    if not icds:
        diagnostic_print('Aucun descripteur Vulkan trouvé dans les chemins inspectés.')
    for icd in sorted(icds):
        diagnostic_print(str(icd))
        try:
            diagnostic_print(json.dumps(json.loads(icd.read_text()), indent=2))
        except (OSError, json.JSONDecodeError) as error:
            diagnostic_print('ICD illisible ou invalide : '+str(error))
    loader_env = dict(os.environ, VK_LOADER_DEBUG='error,warn,driver')
    if DIAGNOSTIC_LD_DEBUG:
        loader_env['LD_DEBUG'] = 'libs'
    diagnostic_command(['vulkaninfo', '--summary'], child_env=loader_env, loader_trace=DIAGNOSTIC_LD_DEBUG)
    diagnostic_print('\nRapport local : '+diagnostic_report.name)


### Contrôle Vulkan bloquant
Ne pas poursuivre si cette cellule échoue, même si la cellule de diagnostic a terminé.


In [ ]:
for directory in (Path('/usr/share/vulkan/icd.d'), Path('/etc/vulkan/icd.d')):
    print('ICD :', directory, [p.name for p in directory.glob('*.json')], flush=True)
print('Variables Vulkan :', {k: v for k, v in os.environ.items() if k.startswith('VK_')}, flush=True)
vulkan_log = run(['vulkaninfo', '--summary'], 'vulkan-system')
vulkan_text = vulkan_log.read_text(errors='replace')
if 'NVIDIA' not in vulkan_text or 'deviceType' not in vulkan_text:
    raise RuntimeError('Vulkan ne voit pas NVIDIA. Ne pas lancer Spirula. Examiner le log et les ICD du runtime.')


## 3. Environnement de calcul (long au premier lancement)
Réutilisé si le YAML est inchangé. Une modification du YAML impose un nouvel environnement, pas une réutilisation silencieuse.


In [ ]:
import shutil, tarfile, urllib.request, tempfile
MAMBA = Path('/content/bin/micromamba')
MAMBA_ROOT = Path('/content/micromamba')
MAMBA_ENV = 'gsplat'
os.environ['MAMBA_ROOT_PREFIX'] = str(MAMBA_ROOT)
if not MAMBA.is_file():
    MAMBA.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=WORK) as temp:
        package = Path(temp) / 'micromamba.tar.bz2'
        print('Téléchargement micromamba...', flush=True)
        urllib.request.urlretrieve('https://micro.mamba.pm/api/micromamba/linux-64/latest', package)
        with tarfile.open(package) as tar:
            member = tar.getmember('bin/micromamba')
            if not member.isfile():
                raise RuntimeError('Archive micromamba invalide')
            with tar.extractfile(member) as source, MAMBA.open('wb') as target:
                shutil.copyfileobj(source, target)
    MAMBA.chmod(0o755)
ENV_FILE = PROJECT / 'environment/google.colab/conda_env.yml'
ENV_PREFIX = MAMBA_ROOT / 'envs' / MAMBA_ENV
ENV_STAMP = ENV_PREFIX / '.colab-yaml.sha256'
fingerprint = sha256(ENV_FILE)
if ENV_PREFIX.exists():
    if not ENV_STAMP.is_file() or ENV_STAMP.read_text().strip() != fingerprint:
        raise RuntimeError('Environnement existant non vérifié ou YAML modifié. Choisir un nouveau MAMBA_ENV (ex. gsplat_v2) puis relancer cette cellule.')
    print('Environnement inchangé : réutilisé.', flush=True)
else:
    run([str(MAMBA), 'create', '--yes', '--name', MAMBA_ENV, '--file', str(ENV_FILE)], 'conda-create')
    ENV_STAMP.write_text(fingerprint+'\n')
def in_env(command):
    return [str(MAMBA), 'run', '--no-capture-output', '-n', MAMBA_ENV, *map(str, command)]
run(in_env(['python', '-u', '-c', "import torch, cv2, numpy, PIL, tqdm; print('CUDA:', torch.cuda.is_available(), flush=True); assert torch.cuda.is_available(), 'CUDA indisponible'; print(torch.cuda.get_device_name(0), flush=True)"]), 'python-gpu')
env_vulkan = run(in_env(['vulkaninfo', '--summary']), 'vulkan-conda')
if 'NVIDIA' not in env_vulkan.read_text(errors='replace'):
    raise RuntimeError('Vulkan NVIDIA absent dans Conda : vérifier ses bibliothèques et variables VK_* avant le calcul.')


## 4. Spirula précompilé et vidéo locale
Archive choisie explicitement, extraction locale réutilisée par SHA256. Les ressources et bibliothèques restent à côté du binaire. La vidéo est copiée une seule fois et vérifiée ; une vidéo différente exige un nouveau RUN_NAME.


In [ ]:
SPIRULA_ROOT, SPIRULA_BIN = restore_spirula(SPIRULA_ARCHIVE, WORK / 'assets', SPIRULA_BINARY_RELATIVE)
SAM_MODEL = SPIRULA_ROOT / SAM_MODEL_RELATIVE
if not SAM_MODEL.is_file():
    raise FileNotFoundError(f'Checkpoint SAM absent : {SAM_MODEL}')
libraries = run(in_env(['ldd', str(SPIRULA_BIN)]), 'spirula-libraries')
if 'not found' in libraries.read_text(errors='replace'):
    raise RuntimeError('Bibliothèques Spirula absentes : voir le log ldd ; archive/runtime incompatibles.')
VIDEO_LOCAL = RUN / ('input'+VIDEO_DRIVE.suffix)
video_hash = sha256(VIDEO_DRIVE)
if VIDEO_LOCAL.exists():
    if sha256(VIDEO_LOCAL) != video_hash:
        raise RuntimeError('Vidéo locale différente/incomplète : choisir un nouveau RUN_NAME.')
    print('Vidéo locale inchangée : réutilisée.', flush=True)
else:
    print('Copie vidéo Drive -> disque local...', flush=True)
    temporary_video = VIDEO_LOCAL.with_suffix(VIDEO_LOCAL.suffix+'.partial')
    shutil.copyfile(VIDEO_DRIVE, temporary_video)
    if sha256(temporary_video) != video_hash:
        raise IOError('Copie vidéo non conforme ; fichier .partial conservé pour diagnostic.')
    temporary_video.rename(VIDEO_LOCAL)
print('Spirula :', SPIRULA_BIN, '\nSAM :', SAM_MODEL, '\nVidéo locale :', VIDEO_LOCAL)


## 5. Réglages et licence
Lire la licence SAM : https://github.com/facebookresearch/sam3/blob/main/LICENSE puis renseigner ACCEPT_SAM_LICENSE selon votre décision. Le notebook ne l'accepte pas à votre place.
OUTPUT_DIR est strictement local. PIPELINE_RESUME ne permet pas de changer images/modèle/paramètres dans un même workspace. Les frames sont extraites par le script vidéo, jamais séparément ici.


In [ ]:
import shlex
if ACCEPT_SAM_LICENSE is not True:
    raise RuntimeError('Lire la licence SAM puis définir ACCEPT_SAM_LICENSE=True si vous l’acceptez.')
SETTINGS = {
    'PIPELINE_RESUME': 'true', 'SPIRULA_ROOT': str(SPIRULA_ROOT), 'SPIRULA_BIN': str(SPIRULA_BIN),
    'PYTHON_BIN': 'python3', 'SFM_IMAGES_DIR': str(FRAMES), 'SFM_IMAGES_RECURSIVE': 'true',
    'OUTPUT_DIR': str(OUTPUT), 'COLMAP_METHOD': 'spirula', 'TRAIN_BACKEND': 'spirula',
    'SFM_MASKS': 'on', 'TRAIN_MASKS': 'on', 'TRAIN_MASKS_BACKEND': 'spirula',
    'TRAIN_MASKS_SPIRULA_MODEL': str(SAM_MODEL), 'TRAIN_MASKS_LABELS': 'sky,clouds',
    'TRAIN_MASKS_MODE': 'empty', 'TRAIN_MASKS_SPIRULA_DEVICE': 'auto',
    'MASKS_JOBS': '1', 'MASKS_ACCEPT_LICENSE': 'true',
    'SFM_MAX_IMAGE_SIZE': '2048', 'SFM_ALIKED_MAX_FEATURES': '4096', 'SFM_THREADS': '6',
    'SFM_SP_QUALITY': 'medium', 'SFM_SP_PAIRS': 'prefilter',
    'SPLIT_NUM_BLOCKS': '2', 'SPLIT_OVERLAP_MAX_FRACTION': '0.10',
    'TRAIN_NUM_ITER': '12000', 'TRAIN_SP_PRESET': '3dgs', 'TRAIN_SP_QUALITY': 'high',
    'TRAIN_SP_CAP_MAX': '500000', 'TRAIN_SP_REFINE_STOP_ITER': '10000',
    'TRAIN_SP_REFINE_STOP_NUM_ITER': '2000', 'TRAIN_SP_FLOATER_SUPPRESSION': 'mild', 'TRAIN_CLEAN': 'false',
    'POSTPROCESS_SPARSE_BBOX_MODE': 'apply', 'POSTPROCESS_MAX_REMOVAL_FRACTION': '0.20',
    'EXPORT_SPLAT_NAME': RUN_NAME+'.splat',
}
CONFIG = RUN / 'colab_video_config.sh'
CONFIG.write_text('#!/usr/bin/env bash\nset -Eeuo pipefail\n'+''.join(f'{key}={shlex.quote(value)}\n' for key, value in SETTINGS.items())+'export PYTHONUNBUFFERED=1\n')
print('Configuration :', CONFIG, '\nPhase :', PHASE, '\nRésultats locaux :', OUTPUT, flush=True)


## 6. Lancer : logs immédiats
L'affichage gère aussi les barres de progression sans saut de ligne. Si un programme natif tamponne lui-même sa sortie, le helper ne peut pas le forcer à émettre plus tôt. Interrompre la cellule arrête le groupe de processus ; les logs ne sont pas écrasés à la relance.


In [ ]:
command = in_env(['bash', 'scripts/videos_to_3dgs.sh', str(CONFIG), '--video', str(VIDEO_LOCAL), '--num-frames', str(NUM_FRAMES), '--frames-dir', str(FRAMES), '--phase', PHASE])
print('Début du calcul local ; aucune écriture intermédiaire sur Drive.', flush=True)
run(command, 'pipeline-'+PHASE, cwd=PROJECT)
print('Phase terminée :', PHASE, flush=True)
if SAVE_ON_SUCCESS:
    save_snapshot(RUN, DRIVE_BACKUPS)


## 7. Sauvegarde manuelle après arrêt/échec
Exécuter seulement quand le calcul est arrêté. Une archive horodatée contient vidéo locale, frames, masques, résultats et logs ; les liens images sont matérialisés pour être lisibles hors Colab. L'archive est créée localement, puis copiée et vérifiée sur Drive en un fichier. Prévoir assez de disque local. Les anciennes sauvegardes restent intactes.
**Ce n'est pas une reprise automatique après perte du runtime** : les chemins/signatures du pipeline doivent être conservés et les liens restaurés avant d'adopter un workspace. Dans le même runtime, PIPELINE_RESUME réutilise le run local. Ne pas déplacer le vieux workspace Drive dans ce run.


In [ ]:
snapshot = save_snapshot(RUN, DRIVE_BACKUPS)


## 8. Résultats et diagnostic (sans lister toutes les images)


In [ ]:
import json
splats = sorted(OUTPUT.rglob('*.splat')) if OUTPUT.exists() else []
for path in splats:
    print(f'{path.stat().st_size/2**20:.1f} Mio  {path}')
if not splats:
    print('Pas encore de splat exporté ; consulter les logs locaux :', LOGS)
report = OUTPUT / 'pipeline_report.json'
if report.is_file():
    for entry in json.loads(report.read_text()).get('runs', [])[-5:]:
        print(entry.get('pipeline'), entry.get('status'), entry.get('duration_seconds'))
